# Participação entre dias — ENEM 2024

Etapa 2.1: seis campos, todos os registros publicados. Presenças LC/CH no dia 1 e CN/MT no dia 2. O cálculo de 2025 é reutilizado, sem regravar seus derivados. Este notebook prepara a entrada, valida e publica indicadores 2024 e consolidação por edição. Contrato: `docs/contrato_participacao_2024_2025.md`.


In [1]:
from pathlib import Path
import sys, json
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/participacao.py').exists())
if str(raiz) not in sys.path: sys.path.insert(0, str(raiz))
import pandas as pd
from IPython.display import display
from src.participacao_entrada import preparar_participacao
from src.participacao_execucao import executar_participacao
from src.participacao_comparacao import executar_comparacao
from src.desempenho_execucao import hash_arquivo
legados = [p for pasta in ['analitica', 'reports', 'apresentacao', 'trusted'] for p in (raiz/pasta).rglob('*') if p.is_file() and '2025' in p.name and '2024' not in p.name]
antes = {p: hash_arquivo(p) for p in legados}
print('Artefatos 2025 protegidos:', len(antes))


Artefatos 2025 protegidos: 50


In [2]:
preparo = preparar_participacao(raiz, ano=2024)
print(json.dumps(preparo, ensure_ascii=False, indent=2))


{
  "edicao": 2024,
  "total_entrada": 4332944,
  "total_saida": 4332944,
  "nulos": {
    "NU_SEQUENCIAL": 0,
    "NU_ANO": 0,
    "TP_PRESENCA_CN": 0,
    "TP_PRESENCA_CH": 0,
    "TP_PRESENCA_LC": 0,
    "TP_PRESENCA_MT": 0
  },
  "codigos_invalidos": {
    "TP_PRESENCA_CN": 0,
    "TP_PRESENCA_CH": 0,
    "TP_PRESENCA_LC": 0,
    "TP_PRESENCA_MT": 0
  },
  "chaves_nulas": 0,
  "duplicadas": 0,
  "anos_invalidos": 0,
  "divergencias_releitura": 0,
  "sha256_raw_antes": "6fc3469bb7b67b02840f0d073021ce58358cb68ef896feb017fd5730cb97b49d",
  "sha256_raw_depois": "6fc3469bb7b67b02840f0d073021ce58358cb68ef896feb017fd5730cb97b49d",
  "parquet": "trusted/participacao_2024_base.parquet",
  "sha256_parquet": "609adbb172a010fa90941650a52ca2216fc64c063a5c78a836579eb71192ac14"
}


In [3]:
resultado, validacao = executar_participacao(raiz, ano=2024)
validacao['preparo'] = preparo
arquivo = raiz/'reports/validacao_participacao_2024.json'
arquivo.write_text(json.dumps(validacao, ensure_ascii=False, indent=2), encoding='utf-8')
print('Controles aprovados:', all(validacao['controles'].values()))
print('Alertas:', validacao['alertas_pares'])
display(pd.DataFrame(resultado['dias']))
display(pd.DataFrame([resultado['resumo']]))


Controles aprovados: True
Alertas: []


,dia,status,quantidade,percentual_base
0,1,presente,3167955,73.113223
1,1,ausente,1159331,26.756196
2,1,eliminado,5658,0.130581
3,1,misto,0,0.000000
4,1,dados_ausentes,0,0.000000
5,1,codigo_invalido,0,0.000000
6,2,presente,3004981,69.351946
7,2,ausente,1325660,30.594903
8,2,eliminado,2303,0.053151
9,2,misto,0,0.000000


,total_base,presentes_dia1,presentes_dia2,presentes_ambos,somente_primeiro_ausente_segundo,somente_segundo_ausente_primeiro,ausentes_ambos,demais_situacoes,retencao_percentual,retencao_denominador,diferenca_liquida_dia2_menos_dia1,presente_dia1_sem_presenca_completa_dia2,presente_dia2_sem_presenca_completa_dia1,pct_presentes_ambos_base,pct_somente_primeiro_ausente_segundo_base,pct_somente_segundo_ausente_primeiro_base,pct_ausentes_ambos_base,pct_demais_situacoes_base
0,4332944,3167955,3004981,2990093,175619,14004,1145272,7956,94.385589,3167955,-162974,177862,14888,69.008346,4.05311,0.323198,26.431729,0.183616


In [4]:
consolidado = executar_comparacao(raiz)
display(pd.DataFrame(consolidado['comparacao']))
alterados = [str(p.relative_to(raiz)) for p,h in antes.items() if hash_arquivo(p) != h]
if alterados: raise RuntimeError(f'Artefatos 2025 alterados: {alterados}')
print('2025 preservado por SHA-256. Comparação de indicadores; sem vínculo entre registros de edições diferentes.')


,indicador,contagem_2024,denominador_2024,taxa_2024,contagem_2025,denominador_2025,taxa_2025,diferenca_contagem_2025_menos_2024,diferenca_pp_2025_menos_2024
0,presenca_dia1,3167955,4332944,73.113223,3457555,4810772,71.871105,289600,-1.242118
1,presenca_dia2,3004981,4332944,69.351946,3260336,4810772,67.771576,255355,-1.580370
2,presenca_ambos,2990093,4332944,69.008346,3244348,4810772,67.439238,254255,-1.569108
3,retencao,2990093,3167955,94.385589,3244348,3457555,93.833590,254255,-0.551999


2025 preservado por SHA-256. Comparação de indicadores; sem vínculo entre registros de edições diferentes.
